# Final Submission: ML Pipeline TFX + Deployment + Monitoring

Klasifikasi tingkat obesitas dari kebiasaan makan dan kondisi fisik.

Pipeline TFX dijalankan dengan orchestrator **Apache Beam** (`BeamDagRunner`) melalui `local_pipeline.py`:
`ExampleGen → StatisticsGen → SchemaGen → ExampleValidator → Transform → Tuner → Trainer → Resolver → Evaluator → Pusher`.

Kode komponen ada di `components.py`; modul preprocessing, tuning, dan pelatihan ada di folder `modules/`.

## 1. Persiapan Library

Colab sudah menyediakan TFX dan TensorFlow. Sel ini hanya memastikan `keras-tuner` tersedia dan menampilkan versi library.

In [ ]:
!pip install -q -U tfx tensorflow-model-analysis keras-tuner
!pip list 2>/dev/null | grep -i -E "^(tfx|tensorflow|keras|keras-tuner|tensorflow-transform|tensorflow_model_analysis|apache-beam) "

## 2. Dataset

**Estimation of Obesity Levels Based on Eating Habits and Physical Condition** (UCI ML Repository, ID 544): ±2.111 responden dengan 16 fitur dan 7 kelas target `NObeyesdad`.

In [ ]:
!mkdir -p data raw
!curl -L -o raw/obesity.zip "https://archive.ics.uci.edu/static/public/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition.zip"
!unzip -o raw/obesity.zip -d raw
!cp raw/*.csv data/
!ls data

In [ ]:
import pandas as pd

df = pd.read_csv("data/ObesityDataSet_raw_and_data_sinthetic.csv")
print(df.shape)
display(df.head())
display(df["NObeyesdad"].value_counts())

## 3. Struktur Kode

| Berkas | Fungsi |
|---|---|
| `modules/features.py` | Konstanta fitur dan label |
| `modules/transform.py` | `preprocessing_fn` untuk komponen Transform |
| `modules/model.py` | Arsitektur model, pemuatan data, penyimpanan model |
| `modules/tuner.py` | `tuner_fn` (KerasTuner RandomSearch) untuk komponen Tuner |
| `modules/trainer.py` | `run_fn` untuk komponen Trainer |
| `components.py` | Definisi seluruh komponen TFX |
| `local_pipeline.py` | Menjalankan pipeline dengan Apache Beam |

Hasil pipeline disimpan di folder `<username_dicoding>-pipeline/`.

## 4. Penjelasan Komponen

- **ExampleGen:** membaca CSV dan membagi data 80% latih / 20% evaluasi.
- **StatisticsGen:** menghitung statistik tiap fitur.
- **SchemaGen:** membuat skema (tipe dan domain fitur).
- **ExampleValidator:** mendeteksi anomali data terhadap skema.
- **Transform:** z-score untuk fitur numerik, one-hot untuk fitur kategorikal, label menjadi integer. `Height` dan `Weight` dibuang karena label diturunkan dari BMI (*target leakage*).
- **Tuner:** `RandomSearch` (10 trial) mencari `units_1`, `units_2`, `dropout`, dan `learning_rate` terbaik berdasarkan akurasi validasi.
- **Trainer:** melatih model memakai hyperparameter terbaik dari Tuner.
- **Resolver:** mengambil model *blessed* terakhir sebagai baseline.
- **Evaluator:** menghitung `SparseCategoricalAccuracy` dengan threshold minimal 0,75.
- **Pusher:** menyimpan model yang di-*bless* ke `serving_model/`.

## 5. Menjalankan Pipeline (Apache Beam)

Sebelum menjalankan, pastikan `USERNAME` di `local_pipeline.py` sesuai username Dicoding.

In [ ]:
!python local_pipeline.py

## 6. Hasil Pipeline

In [ ]:
import glob
import os

from local_pipeline import PIPELINE_DIR, PIPELINE_ROOT, SERVING_MODEL_DIR

print("Komponen yang berjalan:", sorted(os.listdir(PIPELINE_ROOT)))
blessing = glob.glob(f"{PIPELINE_ROOT}/Evaluator/blessing/*/*")
print("Status blessing:", [os.path.basename(b) for b in blessing])

In [ ]:
import tensorflow_model_analysis as tfma

eval_dirs = sorted(glob.glob(f"{PIPELINE_ROOT}/Evaluator/evaluation/*"))
result = tfma.load_eval_result(eval_dirs[-1])
tfma.view.render_slicing_metrics(result)

In [ ]:
# Hyperparameter terbaik dari Tuner
best_hp = glob.glob(f"{PIPELINE_ROOT}/Tuner/best_hyperparameters/*/best_hyperparameters.txt")
print(open(best_hp[-1]).read())

In [ ]:
!find {SERVING_MODEL_DIR} -maxdepth 2

## 7. Ringkasan

Isi sel ini setelah pipeline dijalankan: hyperparameter terbaik, akurasi evaluasi, dan status *blessed*. Salin angka yang sama ke `README.md`.